# 🌿 Plant Disease Detection — YOLOv8 Training
**Target: 96%+ Accuracy** | 38 Classes | 14 Crop Species | 54,000+ Images

This notebook trains a YOLOv8m-cls model on the PlantVillage dataset.
Works on multiple tree/plant types — not just apple trees.

**Steps:**
1. Install dependencies
2. Download PlantVillage dataset
3. Organize into train/val/test
4. Train YOLOv8m-cls
5. Evaluate (target 96%+)
6. Export model

---
⚡ **Set runtime to GPU:** Runtime → Change runtime type → T4 GPU

In [ ]:
# ─── Step 1: Install Dependencies ───────────────────────────
!pip install -q ultralytics>=8.2.0 albumentations>=1.3.0 datasets Pillow
!pip install -q tensorflowjs onnx

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ─── Step 2: Download PlantVillage Dataset ──────────────────
import os, shutil, random, json
from pathlib import Path
from datasets import load_dataset

DATA_DIR = Path('data')
RAW_DIR = DATA_DIR / 'raw' / 'plantvillage'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Download from Hugging Face (no API key needed)
print('📦 Downloading PlantVillage dataset from Hugging Face...')
ds = load_dataset('mohanty/PlantVillage', split='train')
label_names = ds.features['label'].names
print(f'Found {len(label_names)} classes, {len(ds)} images')

# Save images organized by class
for idx, sample in enumerate(ds):
    label_name = label_names[sample['label']]
    clean_name = label_name.replace(' ', '_')
    class_dir = RAW_DIR / clean_name
    class_dir.mkdir(parents=True, exist_ok=True)
    sample['image'].save(class_dir / f'{idx:06d}.jpg')
    if (idx + 1) % 10000 == 0:
        print(f'  Saved {idx+1}/{len(ds)} images...')

total = len(list(RAW_DIR.rglob('*.jpg')))
print(f'\n✅ Downloaded {total} images in {len(label_names)} classes')

In [ ]:
# ─── Step 3: Split into Train / Val / Test ──────────────────
DATASET_DIR = DATA_DIR / 'dataset'
random.seed(42)

VAL_RATIO = 0.15
TEST_RATIO = 0.10

class_dirs = sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])
stats = {'train': 0, 'val': 0, 'test': 0}

for class_dir in class_dirs:
    images = list(class_dir.glob('*.jpg'))
    if not images:
        continue
    random.shuffle(images)
    n = len(images)
    n_val = max(1, int(n * VAL_RATIO))
    n_test = max(1, int(n * TEST_RATIO))
    splits = {
        'train': images[:n - n_val - n_test],
        'val': images[n - n_val - n_test:n - n_test],
        'test': images[n - n_test:],
    }
    for split, imgs in splits.items():
        dest = DATASET_DIR / split / class_dir.name
        dest.mkdir(parents=True, exist_ok=True)
        for img in imgs:
            shutil.copy2(img, dest / img.name)
        stats[split] += len(imgs)

print('Dataset split:')
for split, count in stats.items():
    print(f'  {split}: {count} images')
print(f'  Total: {sum(stats.values())} images')
print(f'  Classes: {len(class_dirs)}')

In [ ]:
# ─── Step 4: Train YOLOv8m-cls ───────────────────────────────
# This is the core training cell. Expect ~30-60 min on Colab T4.
#
# Key choices for 96%+ accuracy:
#   - yolov8m-cls: 12M params, strong feature extraction
#   - AdamW + cosine LR: proven for fine-tuning
#   - Strong augmentation: prevents overfitting on lab images
#   - Early stopping: prevents overtraining

from ultralytics import YOLO

# Load pre-trained model (ImageNet weights)
model = YOLO('yolov8m-cls.pt')

# Train with optimized hyperparameters
results = model.train(
    data=str(DATASET_DIR),
    epochs=80,
    imgsz=224,
    batch=64,
    
    # Optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    cos_lr=True,
    warmup_epochs=5,
    
    # Early stopping
    patience=15,
    
    # Data augmentation
    augment=True,
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    degrees=15.0,
    translate=0.1,
    scale=0.3,
    fliplr=0.5,
    flipud=0.2,
    erasing=0.1,
    
    # Output
    project='runs',
    name='plant_disease_v1',
    exist_ok=True,
    plots=True,
    verbose=True,
    seed=42,
)

print('\n✅ Training complete!')

In [ ]:
# ─── Step 5: Evaluate Model ──────────────────────────────────
from ultralytics import YOLO

# Load best weights
best_model = YOLO('runs/plant_disease_v1/weights/best.pt')

# Validation set
print('📊 Validation Results:')
val_results = best_model.val(data=str(DATASET_DIR), split='val', plots=True)
print(f'  Top-1 Accuracy: {val_results.top1 * 100:.2f}%')
print(f'  Top-5 Accuracy: {val_results.top5 * 100:.2f}%')

# Test set
print('\n📊 Test Results:')
test_results = best_model.val(data=str(DATASET_DIR), split='test', plots=True)
print(f'  Top-1 Accuracy: {test_results.top1 * 100:.2f}%')
print(f'  Top-5 Accuracy: {test_results.top5 * 100:.2f}%')

# Check target
if test_results.top1 >= 0.96:
    print(f'\n🎯 TARGET MET! {test_results.top1*100:.2f}% >= 96%')
else:
    print(f'\n⚠️ {test_results.top1*100:.2f}% < 96%. Try yolov8l-cls or more epochs.')

In [ ]:
# ─── Step 5b: If accuracy < 96%, retrain with larger model ──
# Uncomment and run this cell ONLY if Step 5 showed < 96%

# from ultralytics import YOLO
# model_large = YOLO('yolov8l-cls.pt')  # 37M params, higher accuracy
# results = model_large.train(
#     data=str(DATASET_DIR),
#     epochs=100,
#     imgsz=224,
#     batch=32,            # Smaller batch for larger model
#     optimizer='AdamW',
#     lr0=0.0005,          # Lower LR for larger model
#     lrf=0.01,
#     cos_lr=True,
#     warmup_epochs=5,
#     patience=20,
#     augment=True,
#     project='runs',
#     name='plant_disease_v2_large',
#     exist_ok=True,
#     plots=True,
#     seed=42,
# )

In [ ]:
# ─── Step 6: Test Predictions ────────────────────────────────
import matplotlib.pyplot as plt
from PIL import Image
import random

best_model = YOLO('runs/plant_disease_v1/weights/best.pt')

# Get random test images
test_dir = DATASET_DIR / 'test'
test_images = list(test_dir.rglob('*.jpg'))
sample_images = random.sample(test_images, min(9, len(test_images)))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, img_path in zip(axes.flat, sample_images):
    # Run prediction
    result = best_model.predict(str(img_path), verbose=False)[0]
    pred_class = result.names[result.probs.top1]
    pred_conf = result.probs.top1conf.item()
    true_class = img_path.parent.name
    
    # Display
    img = Image.open(img_path)
    ax.imshow(img)
    color = 'green' if pred_class == true_class else 'red'
    ax.set_title(f'Pred: {pred_class}\nTrue: {true_class}\nConf: {pred_conf:.2f}',
                 fontsize=9, color=color)
    ax.axis('off')

plt.tight_layout()
plt.savefig('runs/plant_disease_v1/sample_predictions.png', dpi=150)
plt.show()
print('\n✅ Sample predictions saved!')

In [ ]:
# ─── Step 7: Export Model ────────────────────────────────────
from ultralytics import YOLO

best_model = YOLO('runs/plant_disease_v1/weights/best.pt')

# Export to ONNX (cross-platform, recommended for production)
print('📦 Exporting to ONNX...')
best_model.export(format='onnx', imgsz=224, simplify=True)
print('✅ ONNX export complete')

# Export to TorchScript
print('\n📦 Exporting to TorchScript...')
best_model.export(format='torchscript', imgsz=224)
print('✅ TorchScript export complete')

print('\n📁 Exported files:')
!ls -la runs/plant_disease_v1/weights/

In [ ]:
# ─── Step 8: Download Trained Model ──────────────────────────
# This creates a zip with everything you need
import shutil

# Create a clean export package
export_dir = Path('plant_disease_model')
export_dir.mkdir(exist_ok=True)

# Copy key files
shutil.copy2('runs/plant_disease_v1/weights/best.pt', export_dir / 'best.pt')
shutil.copy2('runs/plant_disease_v1/weights/best.onnx', export_dir / 'best.onnx')

# Save class names
best_model = YOLO('runs/plant_disease_v1/weights/best.pt')
class_names = best_model.names
with open(export_dir / 'class_names.json', 'w') as f:
    json.dump(class_names, f, indent=2)

# Copy training plots
for plot_file in Path('runs/plant_disease_v1').glob('*.png'):
    shutil.copy2(plot_file, export_dir / plot_file.name)

# Create zip
shutil.make_archive('plant_disease_model', 'zip', '.', 'plant_disease_model')

# Download (works in Colab)
try:
    from google.colab import files
    files.download('plant_disease_model.zip')
    print('\n✅ Download started! Check your browser downloads.')
except:
    print(f'\n✅ Model package ready at: plant_disease_model.zip')
    print('   Download it manually from the file browser on the left.')

---
## 📋 Summary

| Item | Details |
|------|--------|
| Model | YOLOv8m-cls (fine-tuned) |
| Dataset | PlantVillage (54K+ images) |
| Classes | 38 (14 crop species) |
| Target | 96%+ Top-1 Accuracy |
| Export | .pt, .onnx, .torchscript |

### Next Steps
1. Download `plant_disease_model.zip`
2. Place `best.pt` in your app's model directory
3. Use class_names.json to map predictions to disease names

### If accuracy < 96%:
- Run Step 5b (train with `yolov8l-cls` — larger model)
- Increase epochs to 120
- Add more augmentation

---
*Open source — MIT License*